# Generación completa del dataset sintético de mora bancaria

Este notebook genera desde cero una base sintética de **21.500 registros y 35 variables**, adaptada al contexto bancario ecuatoriano. `ID` se encuentra protegido con terminación `XXX`, `fecha_reporte` ocupa la segunda columna y contiene 24 cierres mensuales equilibrados, mientras que `mora` es textual y permanece como última variable.

Todas las líneas ejecutables incluyen una explicación previa en español. No se requiere cargar ningún archivo para generar la base.


## Características conservadas

- Zonas donde Banco ProCredit mantiene presencia: Quito, Guayaquil, Ambato, Ibarra, Cuenca y Santo Domingo.
- Vivienda únicamente `PROPIA` o `ARRENDADA`.
- Coherencia entre días de atraso, historial crediticio, clasificación SBS y nivel textual de mora.
- `tipo_credito`, `nivel_ahorro`, `deuda_sf`, `zona`, `saldo_pendiente_credito`, `ID`, `fecha_reporte` y `mora` completos.
- Valores faltantes controlados en las demás variables.
- Fechas mensuales desde septiembre de 2024 hasta agosto de 2026.
- Identificadores seudónimos únicos con formato `0000000XXX`.


## 1. Configuración


In [1]:
# Línea 0001: Importa el recurso específico necesario para esta etapa del proceso.
from pathlib import Path

# Línea 0002: Importa la librería necesaria para ejecutar esta operación.
import numpy as np
# Línea 0003: Importa la librería necesaria para ejecutar esta operación.
import pandas as pd


# Línea 0004: Calcula o asigna `N_FILAS` para utilizarlo en los pasos siguientes.
N_FILAS = 21_500
# Línea 0005: Calcula o asigna `SEMILLA` para utilizarlo en los pasos siguientes.
SEMILLA = 20260903
# Línea 0006: Calcula o asigna `SBU_2026_USD` para utilizarlo en los pasos siguientes.
SBU_2026_USD = 482.0
# Línea 0007: Calcula o asigna `DESCARGAR_EN_COLAB` para utilizarlo en los pasos siguientes.
DESCARGAR_EN_COLAB = True
# Línea 0008: Calcula o asigna `NOMBRE_SALIDA` para utilizarlo en los pasos siguientes.
NOMBRE_SALIDA = "base_mora_ecuador_21500_final.csv"


# Línea 0009: Calcula o asigna `ETIQUETAS_MORA` para utilizarlo en los pasos siguientes.
ETIQUETAS_MORA = {
    # Línea 0010: Ejecuta esta instrucción como parte del procedimiento descrito.
    0: "AL DÍA",
    # Línea 0011: Ejecuta esta instrucción como parte del procedimiento descrito.
    1: "MORA DE 1 A 7 DÍAS",
    # Línea 0012: Ejecuta esta instrucción como parte del procedimiento descrito.
    2: "MORA DE 8 A 30 DÍAS",
    # Línea 0013: Ejecuta esta instrucción como parte del procedimiento descrito.
    3: "MORA DE 31 A 60 DÍAS",
    # Línea 0014: Ejecuta esta instrucción como parte del procedimiento descrito.
    4: "MORA DE 61 A 90 DÍAS",
    # Línea 0015: Ejecuta esta instrucción como parte del procedimiento descrito.
    5: "MORA DE MÁS DE 90 DÍAS",
# Línea 0016: Cierra la estructura, llamada o colección iniciada anteriormente.
}


# Línea 0017: Calcula o asigna `NIVEL_MORA_POR_TEXTO` para utilizarlo en los pasos siguientes.
NIVEL_MORA_POR_TEXTO = {texto: nivel for nivel, texto in ETIQUETAS_MORA.items()}


# Línea 0018: Calcula o asigna `COLUMNAS_COMPLETAS` para utilizarlo en los pasos siguientes.
COLUMNAS_COMPLETAS = {
    # Línea 0019: Ejecuta esta instrucción como parte del procedimiento descrito.
    "ID",
    # Línea 0020: Ejecuta esta instrucción como parte del procedimiento descrito.
    "fecha_reporte",
    # Línea 0021: Ejecuta esta instrucción como parte del procedimiento descrito.
    "tipo_credito",
    # Línea 0022: Ejecuta esta instrucción como parte del procedimiento descrito.
    "nivel_ahorro",
    # Línea 0023: Ejecuta esta instrucción como parte del procedimiento descrito.
    "deuda_sf",
    # Línea 0024: Ejecuta esta instrucción como parte del procedimiento descrito.
    "zona",
    # Línea 0025: Ejecuta esta instrucción como parte del procedimiento descrito.
    "saldo_pendiente_credito",
    # Línea 0026: Ejecuta esta instrucción como parte del procedimiento descrito.
    "mora",
# Línea 0027: Cierra la estructura, llamada o colección iniciada anteriormente.
}


# Tasas base de ausencia por columna. Se aplican de forma reproducible.
# Algunas variables derivadas también quedan ausentes si falta un insumo.
# Línea 0028: Calcula o asigna `TASAS_FALTANTES` para utilizarlo en los pasos siguientes.
TASAS_FALTANTES = {
    # Línea 0029: Registra el valor correspondiente a `atraso` dentro de esta estructura.
    "atraso": 0.020,
    # Línea 0030: Registra el valor correspondiente a `vivienda` dentro de esta estructura.
    "vivienda": 0.040,
    # Línea 0031: Registra el valor correspondiente a `edad` dentro de esta estructura.
    "edad": 0.010,
    # Línea 0032: Registra el valor correspondiente a `dias_lab` dentro de esta estructura.
    "dias_lab": 0.060,
    # Línea 0033: Registra el valor correspondiente a `exp_sf` dentro de esta estructura.
    "exp_sf": 0.080,
    # Línea 0034: Registra el valor correspondiente a `nivel_ahorro` dentro de esta estructura.
    "nivel_ahorro": 0.000,
    # Línea 0035: Registra el valor correspondiente a `ingreso` dentro de esta estructura.
    "ingreso": 0.040,
    # Línea 0036: Registra el valor correspondiente a `linea_sf` dentro de esta estructura.
    "linea_sf": 0.090,
    # Línea 0037: Registra el valor correspondiente a `deuda_sf` dentro de esta estructura.
    "deuda_sf": 0.000,
    # Línea 0038: Registra el valor correspondiente a `score` dentro de esta estructura.
    "score": 0.030,
    # Línea 0039: Registra el valor correspondiente a `zona` dentro de esta estructura.
    "zona": 0.000,
    # Línea 0040: Registra el valor correspondiente a `clasif_sbs` dentro de esta estructura.
    "clasif_sbs": 0.040,
    # Línea 0041: Registra el valor correspondiente a `nivel_educ` dentro de esta estructura.
    "nivel_educ": 0.050,
    # Línea 0042: Registra el valor correspondiente a `mora_max_3m` dentro de esta estructura.
    "mora_max_3m": 0.050,
    # Línea 0043: Registra el valor correspondiente a `mora_max_6m` dentro de esta estructura.
    "mora_max_6m": 0.060,
    # Línea 0044: Registra el valor correspondiente a `mora_max_12m` dentro de esta estructura.
    "mora_max_12m": 0.070,
    # Línea 0045: Registra el valor correspondiente a `atrasos_6m_12m` dentro de esta estructura.
    "atrasos_6m_12m": 0.060,
    # Línea 0046: Registra el valor correspondiente a `atrasos_12m` dentro de esta estructura.
    "atrasos_12m": 0.070,
    # Línea 0047: Registra el valor correspondiente a `dti` dentro de esta estructura.
    "dti": 0.050,
    # Línea 0048: Registra el valor correspondiente a `cuota_ingreso` dentro de esta estructura.
    "cuota_ingreso": 0.060,
    # Línea 0049: Registra el valor correspondiente a `saldo_promedio_3m` dentro de esta estructura.
    "saldo_promedio_3m": 0.090,
    # Línea 0050: Registra el valor correspondiente a `variacion_depositos_ingresos` dentro de esta estructura.
    "variacion_depositos_ingresos": 0.120,
    # Línea 0051: Registra el valor correspondiente a `num_entidades_deuda` dentro de esta estructura.
    "num_entidades_deuda": 0.050,
    # Línea 0052: Registra el valor correspondiente a `nuevos_creditos_3m_6m` dentro de esta estructura.
    "nuevos_creditos_3m_6m": 0.070,
    # Línea 0053: Registra el valor correspondiente a `nuevos_creditos_6m` dentro de esta estructura.
    "nuevos_creditos_6m": 0.070,
    # Línea 0054: Registra el valor correspondiente a `porcentaje_utilizacion_linea` dentro de esta estructura.
    "porcentaje_utilizacion_linea": 0.060,
    # Línea 0055: Registra el valor correspondiente a `saldo_pendiente_credito` dentro de esta estructura.
    "saldo_pendiente_credito": 0.000,
    # Línea 0056: Registra el valor correspondiente a `cuotas_restantes` dentro de esta estructura.
    "cuotas_restantes": 0.040,
    # Línea 0057: Registra el valor correspondiente a `tipo_credito` dentro de esta estructura.
    "tipo_credito": 0.000,
    # Línea 0058: Registra el valor correspondiente a `antiguedad_cliente_banco` dentro de esta estructura.
    "antiguedad_cliente_banco": 0.060,
    # Línea 0059: Registra el valor correspondiente a `numero_productos_banco` dentro de esta estructura.
    "numero_productos_banco": 0.040,
    # Línea 0060: Registra el valor correspondiente a `reestructuraciones_refinanciamientos_previos` dentro de esta estructura.
    "reestructuraciones_refinanciamientos_previos": 0.030,
# Línea 0061: Cierra la estructura, llamada o colección iniciada anteriormente.
}


# Línea 0062: Calcula o asigna `COLUMNAS` para utilizarlo en los pasos siguientes.
COLUMNAS = [
    # Línea 0063: Ejecuta esta instrucción como parte del procedimiento descrito.
    "ID",
    # Línea 0064: Ejecuta esta instrucción como parte del procedimiento descrito.
    "fecha_reporte",
    # Línea 0065: Ejecuta esta instrucción como parte del procedimiento descrito.
    "atraso",
    # Línea 0066: Ejecuta esta instrucción como parte del procedimiento descrito.
    "vivienda",
    # Línea 0067: Ejecuta esta instrucción como parte del procedimiento descrito.
    "edad",
    # Línea 0068: Ejecuta esta instrucción como parte del procedimiento descrito.
    "dias_lab",
    # Línea 0069: Ejecuta esta instrucción como parte del procedimiento descrito.
    "exp_sf",
    # Línea 0070: Ejecuta esta instrucción como parte del procedimiento descrito.
    "nivel_ahorro",
    # Línea 0071: Ejecuta esta instrucción como parte del procedimiento descrito.
    "ingreso",
    # Línea 0072: Ejecuta esta instrucción como parte del procedimiento descrito.
    "linea_sf",
    # Línea 0073: Ejecuta esta instrucción como parte del procedimiento descrito.
    "deuda_sf",
    # Línea 0074: Ejecuta esta instrucción como parte del procedimiento descrito.
    "score",
    # Línea 0075: Ejecuta esta instrucción como parte del procedimiento descrito.
    "zona",
    # Línea 0076: Ejecuta esta instrucción como parte del procedimiento descrito.
    "clasif_sbs",
    # Línea 0077: Ejecuta esta instrucción como parte del procedimiento descrito.
    "nivel_educ",
    # Línea 0078: Ejecuta esta instrucción como parte del procedimiento descrito.
    "mora_max_3m",
    # Línea 0079: Ejecuta esta instrucción como parte del procedimiento descrito.
    "mora_max_6m",
    # Línea 0080: Ejecuta esta instrucción como parte del procedimiento descrito.
    "mora_max_12m",
    # Línea 0081: Ejecuta esta instrucción como parte del procedimiento descrito.
    "atrasos_6m_12m",
    # Línea 0082: Ejecuta esta instrucción como parte del procedimiento descrito.
    "atrasos_12m",
    # Línea 0083: Ejecuta esta instrucción como parte del procedimiento descrito.
    "dti",
    # Línea 0084: Ejecuta esta instrucción como parte del procedimiento descrito.
    "cuota_ingreso",
    # Línea 0085: Ejecuta esta instrucción como parte del procedimiento descrito.
    "saldo_promedio_3m",
    # Línea 0086: Ejecuta esta instrucción como parte del procedimiento descrito.
    "variacion_depositos_ingresos",
    # Línea 0087: Ejecuta esta instrucción como parte del procedimiento descrito.
    "num_entidades_deuda",
    # Línea 0088: Ejecuta esta instrucción como parte del procedimiento descrito.
    "nuevos_creditos_3m_6m",
    # Línea 0089: Ejecuta esta instrucción como parte del procedimiento descrito.
    "nuevos_creditos_6m",
    # Línea 0090: Ejecuta esta instrucción como parte del procedimiento descrito.
    "porcentaje_utilizacion_linea",
    # Línea 0091: Ejecuta esta instrucción como parte del procedimiento descrito.
    "saldo_pendiente_credito",
    # Línea 0092: Ejecuta esta instrucción como parte del procedimiento descrito.
    "cuotas_restantes",
    # Línea 0093: Ejecuta esta instrucción como parte del procedimiento descrito.
    "tipo_credito",
    # Línea 0094: Ejecuta esta instrucción como parte del procedimiento descrito.
    "antiguedad_cliente_banco",
    # Línea 0095: Ejecuta esta instrucción como parte del procedimiento descrito.
    "numero_productos_banco",
    # Línea 0096: Ejecuta esta instrucción como parte del procedimiento descrito.
    "reestructuraciones_refinanciamientos_previos",
    # Línea 0097: Ejecuta esta instrucción como parte del procedimiento descrito.
    "mora",
# Línea 0098: Cierra la estructura, llamada o colección iniciada anteriormente.
]


# Línea 0099: Calcula o asigna `COLUMNAS_ENTERAS_CON_FALTANTES` para utilizarlo en los pasos siguientes.
COLUMNAS_ENTERAS_CON_FALTANTES = [
    # Línea 0100: Ejecuta esta instrucción como parte del procedimiento descrito.
    "atraso",
    # Línea 0101: Ejecuta esta instrucción como parte del procedimiento descrito.
    "edad",
    # Línea 0102: Ejecuta esta instrucción como parte del procedimiento descrito.
    "dias_lab",
    # Línea 0103: Ejecuta esta instrucción como parte del procedimiento descrito.
    "exp_sf",
    # Línea 0104: Ejecuta esta instrucción como parte del procedimiento descrito.
    "score",
    # Línea 0105: Ejecuta esta instrucción como parte del procedimiento descrito.
    "mora_max_3m",
    # Línea 0106: Ejecuta esta instrucción como parte del procedimiento descrito.
    "mora_max_6m",
    # Línea 0107: Ejecuta esta instrucción como parte del procedimiento descrito.
    "mora_max_12m",
    # Línea 0108: Ejecuta esta instrucción como parte del procedimiento descrito.
    "atrasos_6m_12m",
    # Línea 0109: Ejecuta esta instrucción como parte del procedimiento descrito.
    "atrasos_12m",
    # Línea 0110: Ejecuta esta instrucción como parte del procedimiento descrito.
    "num_entidades_deuda",
    # Línea 0111: Ejecuta esta instrucción como parte del procedimiento descrito.
    "nuevos_creditos_3m_6m",
    # Línea 0112: Ejecuta esta instrucción como parte del procedimiento descrito.
    "nuevos_creditos_6m",
    # Línea 0113: Ejecuta esta instrucción como parte del procedimiento descrito.
    "cuotas_restantes",
    # Línea 0114: Ejecuta esta instrucción como parte del procedimiento descrito.
    "antiguedad_cliente_banco",
    # Línea 0115: Ejecuta esta instrucción como parte del procedimiento descrito.
    "numero_productos_banco",
# Línea 0116: Cierra la estructura, llamada o colección iniciada anteriormente.
]


# Línea 0117: Calcula o asigna `COLUMNAS_MONETARIAS` para utilizarlo en los pasos siguientes.
COLUMNAS_MONETARIAS = [
    # Línea 0118: Ejecuta esta instrucción como parte del procedimiento descrito.
    "ingreso",
    # Línea 0119: Ejecuta esta instrucción como parte del procedimiento descrito.
    "linea_sf",
    # Línea 0120: Ejecuta esta instrucción como parte del procedimiento descrito.
    "deuda_sf",
    # Línea 0121: Ejecuta esta instrucción como parte del procedimiento descrito.
    "saldo_promedio_3m",
    # Línea 0122: Ejecuta esta instrucción como parte del procedimiento descrito.
    "saldo_pendiente_credito",
# Línea 0123: Cierra la estructura, llamada o colección iniciada anteriormente.
]


# Línea 0124: Calcula o asigna `COLUMNAS_PORCENTAJE` para utilizarlo en los pasos siguientes.
COLUMNAS_PORCENTAJE = [
    # Línea 0125: Ejecuta esta instrucción como parte del procedimiento descrito.
    "dti",
    # Línea 0126: Ejecuta esta instrucción como parte del procedimiento descrito.
    "cuota_ingreso",
    # Línea 0127: Ejecuta esta instrucción como parte del procedimiento descrito.
    "variacion_depositos_ingresos",
    # Línea 0128: Ejecuta esta instrucción como parte del procedimiento descrito.
    "porcentaje_utilizacion_linea",
# Línea 0129: Cierra la estructura, llamada o colección iniciada anteriormente.
]


## 2. Funciones auxiliares


In [2]:
# Línea 0001: Define la función `_sigmoide` para reutilizar este procedimiento de forma ordenada.
def _sigmoide(x):
    # Línea 0002: Devuelve el resultado calculado por la función.
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


# Línea 0003: Define la función `_muestrear_por_fila` para reutilizar este procedimiento de forma ordenada.
def _muestrear_por_fila(rng, categorias, probabilidades):
    # Línea 0004: Documenta la finalidad del bloque o de la función.
    """Muestrea una categoría por fila desde una matriz de probabilidades."""
    # Línea 0005: Calcula o asigna `probabilidades` para utilizarlo en los pasos siguientes.
    probabilidades = np.asarray(probabilidades, dtype=float)
    # Línea 0006: Calcula o asigna `probabilidades` para utilizarlo en los pasos siguientes.
    probabilidades = probabilidades / probabilidades.sum(axis=1, keepdims=True)
    # Línea 0007: Calcula o asigna `acumuladas` para utilizarlo en los pasos siguientes.
    acumuladas = np.cumsum(probabilidades, axis=1)
    # Línea 0008: Calcula o asigna `u` para utilizarlo en los pasos siguientes.
    u = rng.random(probabilidades.shape[0])[:, None]
    # Línea 0009: Calcula o asigna `indices` para utilizarlo en los pasos siguientes.
    indices = (u > acumuladas).sum(axis=1)
    # Línea 0010: Devuelve el resultado calculado por la función.
    return np.asarray(categorias, dtype=object)[indices]


# Línea 0011: Define la función `_dias_segun_clase` para reutilizar este procedimiento de forma ordenada.
def _dias_segun_clase(rng, clases):
    # Línea 0012: Documenta la finalidad del bloque o de la función.
    """Convierte clases de mora 0-5 a días dentro de sus intervalos."""
    # Línea 0013: Calcula o asigna `clases` para utilizarlo en los pasos siguientes.
    clases = np.asarray(clases, dtype=int)
    # Línea 0014: Calcula o asigna `dias` para utilizarlo en los pasos siguientes.
    dias = np.zeros(len(clases), dtype=int)
    # Línea 0015: Calcula o asigna `rangos` para utilizarlo en los pasos siguientes.
    rangos = {1: (1, 7), 2: (8, 30), 3: (31, 60), 4: (61, 90), 5: (91, 240)}
    # Línea 0016: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for clase, (minimo, maximo) in rangos.items():
        # Línea 0017: Ejecuta esta instrucción como parte del procedimiento descrito.
        mascara = clases == clase
        # Línea 0018: Calcula o asigna `dias[mascara]` para utilizarlo en los pasos siguientes.
        dias[mascara] = rng.integers(minimo, maximo + 1, mascara.sum())
    # Línea 0019: Devuelve el resultado calculado por la función.
    return dias


# Línea 0020: Define la función `_asignar_mora_ordenada` para reutilizar este procedimiento de forma ordenada.
def _asignar_mora_ordenada(riesgo):
    # Línea 0021: Documenta la finalidad del bloque o de la función.
    """Asigna una mora desbalanceada y ordenada a partir del riesgo latente."""
    # Línea 0022: Calcula o asigna `proporciones` para utilizarlo en los pasos siguientes.
    proporciones = np.array([0.720, 0.100, 0.070, 0.045, 0.035, 0.030])
    # Línea 0023: Calcula o asigna `conteos` para utilizarlo en los pasos siguientes.
    conteos = np.floor(proporciones * len(riesgo)).astype(int)
    # Línea 0024: Calcula o asigna `conteos[0] +` para utilizarlo en los pasos siguientes.
    conteos[0] += len(riesgo) - conteos.sum()
    # Línea 0025: Calcula o asigna `orden` para utilizarlo en los pasos siguientes.
    orden = np.argsort(riesgo, kind="mergesort")
    # Línea 0026: Calcula o asigna `mora` para utilizarlo en los pasos siguientes.
    mora = np.empty(len(riesgo), dtype=np.int8)
    # Línea 0027: Calcula o asigna `inicio` para utilizarlo en los pasos siguientes.
    inicio = 0
    # Línea 0028: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for clase, cantidad in enumerate(conteos):
        # Línea 0029: Calcula o asigna `seleccion` para utilizarlo en los pasos siguientes.
        seleccion = orden[inicio : inicio + cantidad]
        # Línea 0030: Calcula o asigna `mora[seleccion]` para utilizarlo en los pasos siguientes.
        mora[seleccion] = clase
        # Línea 0031: Calcula o asigna `inicio +` para utilizarlo en los pasos siguientes.
        inicio += cantidad
    # Línea 0032: Devuelve el resultado calculado por la función.
    return mora


# Línea 0033: Define la función `_clasificacion_sbs` para reutilizar este procedimiento de forma ordenada.
def _clasificacion_sbs(rng, mora):
    # Línea 0034: Documenta la finalidad del bloque o de la función.
    """Genera una clasificación A1-E compatible con cada tramo de mora."""
    # Línea 0035: Calcula o asigna `opciones` para utilizarlo en los pasos siguientes.
    opciones = {
        # Línea 0036: Ejecuta esta instrucción como parte del procedimiento descrito.
        0: (np.array(["A1", "A2", "A3"]), [0.50, 0.32, 0.18]),
        # Línea 0037: Ejecuta esta instrucción como parte del procedimiento descrito.
        1: (np.array(["A3", "B1"]), [0.45, 0.55]),
        # Línea 0038: Ejecuta esta instrucción como parte del procedimiento descrito.
        2: (np.array(["B1", "B2"]), [0.30, 0.70]),
        # Línea 0039: Ejecuta esta instrucción como parte del procedimiento descrito.
        3: (np.array(["C1", "C2"]), [0.38, 0.62]),
        # Línea 0040: Ejecuta esta instrucción como parte del procedimiento descrito.
        4: (np.array(["D"]), [1.00]),
        # Línea 0041: Ejecuta esta instrucción como parte del procedimiento descrito.
        5: (np.array(["E"]), [1.00]),
    # Línea 0042: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0043: Calcula o asigna `resultado` para utilizarlo en los pasos siguientes.
    resultado = np.empty(len(mora), dtype=object)
    # Línea 0044: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for clase, (categorias, probabilidades) in opciones.items():
        # Línea 0045: Ejecuta esta instrucción como parte del procedimiento descrito.
        mascara = mora == clase
        # Línea 0046: Calcula o asigna `resultado[mascara]` para utilizarlo en los pasos siguientes.
        resultado[mascara] = rng.choice(
            # Línea 0047: Calcula o asigna `categorias, size` para utilizarlo en los pasos siguientes.
            categorias, size=mascara.sum(), p=probabilidades
        # Línea 0048: Cierra la estructura, llamada o colección iniciada anteriormente.
        )
    # Línea 0049: Devuelve el resultado calculado por la función.
    return resultado


# Línea 0050: Define la función `_digito_verificador_cedula` para reutilizar este procedimiento de forma ordenada.
def _digito_verificador_cedula(primeros_nueve):
    # Línea 0051: Documenta la finalidad del bloque o de la función.
    """Calcula el décimo dígito de la cédula ecuatoriana (módulo 10)."""
    # Línea 0052: Calcula o asigna `coeficientes` para utilizarlo en los pasos siguientes.
    coeficientes = (2, 1, 2, 1, 2, 1, 2, 1, 2)
    # Línea 0053: Calcula o asigna `total` para utilizarlo en los pasos siguientes.
    total = 0
    # Línea 0054: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for digito, coeficiente in zip(map(int, primeros_nueve), coeficientes):
        # Línea 0055: Calcula o asigna `producto` para utilizarlo en los pasos siguientes.
        producto = digito * coeficiente
        # Línea 0056: Ejecuta esta instrucción como parte del procedimiento descrito.
        total += producto - 9 if producto >= 10 else producto
    # Línea 0057: Devuelve el resultado calculado por la función.
    return (10 - total % 10) % 10


# Línea 0058: Define la función `_generar_cedulas_ecuatorianas` para reutilizar este procedimiento de forma ordenada.
def _generar_cedulas_ecuatorianas(rng, zonas):
    # Línea 0059: Documenta la finalidad del bloque o de la función.
    """Genera cédulas únicas y válidas, alineando provincia de emisión y zona."""
    # Línea 0060: Calcula o asigna `codigo_provincia` para utilizarlo en los pasos siguientes.
    codigo_provincia = {
        # Línea 0061: Registra el valor correspondiente a `CUENCA` dentro de esta estructura.
        "CUENCA": "01",
        # Línea 0062: Registra el valor correspondiente a `GUAYAQUIL` dentro de esta estructura.
        "GUAYAQUIL": "09",
        # Línea 0063: Registra el valor correspondiente a `IBARRA` dentro de esta estructura.
        "IBARRA": "10",
        # Línea 0064: Registra el valor correspondiente a `QUITO` dentro de esta estructura.
        "QUITO": "17",
        # Línea 0065: Registra el valor correspondiente a `AMBATO` dentro de esta estructura.
        "AMBATO": "18",
        # Línea 0066: Registra el valor correspondiente a `SANTO DOMINGO` dentro de esta estructura.
        "SANTO DOMINGO": "23",
    # Línea 0067: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0068: Calcula o asigna `cedulas` para utilizarlo en los pasos siguientes.
    cedulas = []
    # Línea 0069: Calcula o asigna `usadas` para utilizarlo en los pasos siguientes.
    usadas = set()
    # Línea 0070: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for zona in zonas:
        # Línea 0071: Repite el bloque hasta encontrar un valor que cumpla las reglas establecidas.
        while True:
            # Línea 0072: Calcula o asigna `tercer_digito` para utilizarlo en los pasos siguientes.
            tercer_digito = int(rng.integers(0, 6))
            # Línea 0073: Calcula o asigna `consecutivo` para utilizarlo en los pasos siguientes.
            consecutivo = int(rng.integers(0, 1_000_000))
            # Línea 0074: Calcula o asigna `primeros_nueve` para utilizarlo en los pasos siguientes.
            primeros_nueve = (
                # Línea 0075: Ejecuta esta instrucción como parte del procedimiento descrito.
                f"{codigo_provincia[zona]}{tercer_digito}{consecutivo:06d}"
            # Línea 0076: Cierra la estructura, llamada o colección iniciada anteriormente.
            )
            # Línea 0077: Calcula o asigna `cedula` para utilizarlo en los pasos siguientes.
            cedula = primeros_nueve + str(
                # Línea 0078: Ejecuta esta instrucción como parte del procedimiento descrito.
                _digito_verificador_cedula(primeros_nueve)
            # Línea 0079: Cierra la estructura, llamada o colección iniciada anteriormente.
            )
            # Línea 0080: Evalúa esta condición antes de ejecutar el bloque asociado.
            if cedula not in usadas:
                # Línea 0081: Ejecuta esta instrucción como parte del procedimiento descrito.
                usadas.add(cedula)
                # Línea 0082: Ejecuta esta instrucción como parte del procedimiento descrito.
                cedulas.append(cedula)
                # Línea 0083: Ejecuta esta instrucción como parte del procedimiento descrito.
                break
    # Línea 0084: Devuelve el resultado calculado por la función.
    return np.asarray(cedulas, dtype=object)


# Línea 0085: Define la función `_validar_cedula_ecuatoriana` para reutilizar este procedimiento de forma ordenada.
def _validar_cedula_ecuatoriana(cedula):
    # Línea 0086: Documenta la finalidad del bloque o de la función.
    """Valida longitud, provincia, persona natural y dígito verificador."""
    # Línea 0087: Calcula o asigna `texto` para utilizarlo en los pasos siguientes.
    texto = str(cedula).zfill(10)
    # Línea 0088: Evalúa esta condición antes de ejecutar el bloque asociado.
    if len(texto) != 10 or not texto.isdigit():
        # Línea 0089: Devuelve el resultado calculado por la función.
        return False
    # Línea 0090: Calcula o asigna `provincia` para utilizarlo en los pasos siguientes.
    provincia = int(texto[:2])
    # Línea 0091: Evalúa esta condición antes de ejecutar el bloque asociado.
    if not 1 <= provincia <= 24 or int(texto[2]) > 5:
        # Línea 0092: Devuelve el resultado calculado por la función.
        return False
    # Línea 0093: Devuelve el resultado calculado por la función.
    return int(texto[-1]) == _digito_verificador_cedula(texto[:9])



# Línea 0094: Define la función `_generar_ids_protegidos` para reutilizar este procedimiento de forma ordenada.
def _generar_ids_protegidos(ids_originales, semilla):
    # Línea 0095: Documenta la finalidad del bloque o de la función.
    """Crea identificadores únicos con siete dígitos visibles y terminación XXX."""
    # Línea 0096: Calcula o asigna `generador` para utilizarlo en los pasos siguientes.
    generador = np.random.default_rng(semilla)
    # Línea 0097: Calcula o asigna `grupos_ecuador` para utilizarlo en los pasos siguientes.
    grupos_ecuador = pd.Series(ids_originales, dtype="string").str[:3]
    # Línea 0098: Calcula o asigna `ids_protegidos` para utilizarlo en los pasos siguientes.
    ids_protegidos = pd.Series(index=grupos_ecuador.index, dtype="string")
    # Línea 0099: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for grupo in sorted(grupos_ecuador.dropna().unique()):
        # Línea 0100: Calcula o asigna `indices_grupo` para utilizarlo en los pasos siguientes.
        indices_grupo = grupos_ecuador[grupos_ecuador.eq(grupo)].index.to_numpy()
        # Línea 0101: Evalúa esta condición antes de ejecutar el bloque asociado.
        if len(indices_grupo) > 10000:
            # Línea 0102: Detiene el proceso y comunica la validación que no fue superada.
            raise ValueError(f"El grupo {grupo} supera la capacidad de anonimización.")
        # Línea 0103: Calcula o asigna `codigos` para utilizarlo en los pasos siguientes.
        codigos = generador.choice(10000, size=len(indices_grupo), replace=False)
        # Línea 0104: Calcula o asigna `codigos_texto` para utilizarlo en los pasos siguientes.
        codigos_texto = [f"{int(codigo):04d}" for codigo in codigos]
        # Línea 0105: Calcula o asigna `ids_protegidos.loc[indices_grupo]` para utilizarlo en los pasos siguientes.
        ids_protegidos.loc[indices_grupo] = [f"{grupo}{codigo}XXX" for codigo in codigos_texto]
    # Línea 0106: Devuelve el resultado calculado por la función.
    return ids_protegidos.to_numpy(dtype=object)


# Línea 0107: Define la función `_generar_fechas_reporte` para reutilizar este procedimiento de forma ordenada.
def _generar_fechas_reporte(n_filas, semilla):
    # Línea 0108: Documenta la finalidad del bloque o de la función.
    """Distribuye 24 cierres mensuales con una diferencia máxima de un registro."""
    # Línea 0109: Calcula o asigna `periodos` para utilizarlo en los pasos siguientes.
    periodos = pd.period_range(start="2024-09", periods=24, freq="M")
    # Línea 0110: Calcula o asigna `fechas` para utilizarlo en los pasos siguientes.
    fechas = periodos.to_timestamp(how="end").normalize().strftime("%Y-%m-%d").tolist()
    # Línea 0111: Calcula o asigna `cantidad_base, sobrantes` para utilizarlo en los pasos siguientes.
    cantidad_base, sobrantes = divmod(n_filas, len(fechas))
    # Línea 0112: Calcula o asigna `fechas_repetidas` para utilizarlo en los pasos siguientes.
    fechas_repetidas = []
    # Línea 0113: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for posicion, fecha in enumerate(fechas):
        # Línea 0114: Calcula o asigna `repeticiones` para utilizarlo en los pasos siguientes.
        repeticiones = cantidad_base + int(posicion < sobrantes)
        # Línea 0115: Ejecuta esta instrucción como parte del procedimiento descrito.
        fechas_repetidas.extend([fecha] * repeticiones)
    # Línea 0116: Calcula o asigna `generador` para utilizarlo en los pasos siguientes.
    generador = np.random.default_rng(semilla)
    # Línea 0117: Devuelve el resultado calculado por la función.
    return generador.permutation(np.array(fechas_repetidas, dtype=object))


## 3. Generación de los registros


In [3]:
# Línea 0001: Define la función `generar_base` para reutilizar este procedimiento de forma ordenada.
def generar_base(n_filas=N_FILAS, semilla=SEMILLA):
    # Línea 0002: Calcula o asigna `rng` para utilizarlo en los pasos siguientes.
    rng = np.random.default_rng(semilla)

    # Ciudades y participación reportadas por Banco ProCredit Ecuador (Q4 2023).
    # Línea 0003: Calcula o asigna `zonas_procredit` para utilizarlo en los pasos siguientes.
    zonas_procredit = np.array(
        # Línea 0004: Ejecuta esta instrucción como parte del procedimiento descrito.
        ["QUITO", "GUAYAQUIL", "AMBATO", "IBARRA", "CUENCA", "SANTO DOMINGO"]
    # Línea 0005: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0006: Calcula o asigna `pesos_zonas` para utilizarlo en los pasos siguientes.
    pesos_zonas = np.array([55.69, 22.49, 12.93, 7.17, 0.89, 0.82])
    # Línea 0007: Calcula o asigna `pesos_zonas /` para utilizarlo en los pasos siguientes.
    pesos_zonas /= pesos_zonas.sum()
    # Línea 0008: Calcula o asigna `zona` para utilizarlo en los pasos siguientes.
    zona = rng.choice(zonas_procredit, n_filas, p=pesos_zonas)

    # Línea 0009: Calcula o asigna `edad` para utilizarlo en los pasos siguientes.
    edad = np.rint(rng.normal(39, 11.5, n_filas)).astype(int)
    # Línea 0010: Calcula o asigna `edad` para utilizarlo en los pasos siguientes.
    edad = np.clip(edad, 18, 78)

    # Línea 0011: Calcula o asigna `niveles_educ` para utilizarlo en los pasos siguientes.
    niveles_educ = np.array(
        # Línea 0012: Ejecuta esta instrucción como parte del procedimiento descrito.
        [
            # Línea 0013: Ejecuta esta instrucción como parte del procedimiento descrito.
            "SIN INSTRUCCIÓN",
            # Línea 0014: Ejecuta esta instrucción como parte del procedimiento descrito.
            "PRIMARIA",
            # Línea 0015: Ejecuta esta instrucción como parte del procedimiento descrito.
            "SECUNDARIA",
            # Línea 0016: Ejecuta esta instrucción como parte del procedimiento descrito.
            "TÉCNICO/TECNOLÓGICO",
            # Línea 0017: Ejecuta esta instrucción como parte del procedimiento descrito.
            "UNIVERSITARIA",
            # Línea 0018: Ejecuta esta instrucción como parte del procedimiento descrito.
            "POSGRADO",
        # Línea 0019: Cierra la estructura, llamada o colección iniciada anteriormente.
        ]
    # Línea 0020: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0021: Calcula o asigna `nivel_educ` para utilizarlo en los pasos siguientes.
    nivel_educ = rng.choice(
        # Línea 0022: Ejecuta esta instrucción como parte del procedimiento descrito.
        niveles_educ,
        # Línea 0023: Ejecuta esta instrucción como parte del procedimiento descrito.
        n_filas,
        # Línea 0024: Calcula o asigna `p` para utilizarlo en los pasos siguientes.
        p=[0.015, 0.180, 0.420, 0.170, 0.190, 0.025],
    # Línea 0025: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Línea 0026: Calcula o asigna `factor_educacion` para utilizarlo en los pasos siguientes.
    factor_educacion = {
        # Línea 0027: Registra el valor correspondiente a `SIN INSTRUCCIÓN` dentro de esta estructura.
        "SIN INSTRUCCIÓN": 0.78,
        # Línea 0028: Registra el valor correspondiente a `PRIMARIA` dentro de esta estructura.
        "PRIMARIA": 0.92,
        # Línea 0029: Registra el valor correspondiente a `SECUNDARIA` dentro de esta estructura.
        "SECUNDARIA": 1.18,
        # Línea 0030: Registra el valor correspondiente a `TÉCNICO/TECNOLÓGICO` dentro de esta estructura.
        "TÉCNICO/TECNOLÓGICO": 1.48,
        # Línea 0031: Registra el valor correspondiente a `UNIVERSITARIA` dentro de esta estructura.
        "UNIVERSITARIA": 2.05,
        # Línea 0032: Registra el valor correspondiente a `POSGRADO` dentro de esta estructura.
        "POSGRADO": 3.00,
    # Línea 0033: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0034: Calcula o asigna `factor_provincia` para utilizarlo en los pasos siguientes.
    factor_provincia = {
        # Línea 0035: Registra el valor correspondiente a `QUITO` dentro de esta estructura.
        "QUITO": 1.15,
        # Línea 0036: Registra el valor correspondiente a `GUAYAQUIL` dentro de esta estructura.
        "GUAYAQUIL": 1.08,
        # Línea 0037: Registra el valor correspondiente a `AMBATO` dentro de esta estructura.
        "AMBATO": 1.02,
        # Línea 0038: Registra el valor correspondiente a `IBARRA` dentro de esta estructura.
        "IBARRA": 0.98,
        # Línea 0039: Registra el valor correspondiente a `CUENCA` dentro de esta estructura.
        "CUENCA": 1.08,
        # Línea 0040: Registra el valor correspondiente a `SANTO DOMINGO` dentro de esta estructura.
        "SANTO DOMINGO": 0.97,
    # Línea 0041: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0042: Calcula o asigna `f_educ` para utilizarlo en los pasos siguientes.
    f_educ = np.array([factor_educacion[x] for x in nivel_educ])
    # Línea 0043: Calcula o asigna `f_prov` para utilizarlo en los pasos siguientes.
    f_prov = np.array([factor_provincia[x] for x in zona])
    # Línea 0044: Calcula o asigna `f_edad` para utilizarlo en los pasos siguientes.
    f_edad = np.clip(0.82 + 0.018 * (edad - 18), 0.82, 1.38)
    # Línea 0045: Calcula o asigna `ingreso` para utilizarlo en los pasos siguientes.
    ingreso = SBU_2026_USD * f_educ * f_prov * f_edad
    # Línea 0046: Calcula o asigna `ingreso *` para utilizarlo en los pasos siguientes.
    ingreso *= rng.lognormal(mean=-0.12, sigma=0.50, size=n_filas)
    # Línea 0047: Calcula o asigna `ingreso` para utilizarlo en los pasos siguientes.
    ingreso = np.round(np.clip(ingreso, 300, 30_000), 2)

    # Línea 0048: Calcula o asigna `prob_vivienda_propia` para utilizarlo en los pasos siguientes.
    prob_vivienda_propia = _sigmoide(
        # Línea 0049: Ejecuta esta instrucción como parte del procedimiento descrito.
        -1.15
        # Línea 0050: Ejecuta esta instrucción como parte del procedimiento descrito.
        + 0.045 * (edad - 35)
        # Línea 0051: Ejecuta esta instrucción como parte del procedimiento descrito.
        + 0.34 * np.log1p(ingreso / SBU_2026_USD)
    # Línea 0052: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0053: Calcula o asigna `vivienda` para utilizarlo en los pasos siguientes.
    vivienda = np.where(
        # Línea 0054: Ejecuta esta instrucción como parte del procedimiento descrito.
        rng.random(n_filas) < prob_vivienda_propia, "PROPIA", "ARRENDADA"
    # Línea 0055: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Línea 0056: Calcula o asigna `max_dias_laborables` para utilizarlo en los pasos siguientes.
    max_dias_laborables = np.maximum(0, (edad - 18) * 365)
    # Línea 0057: Calcula o asigna `dias_lab` para utilizarlo en los pasos siguientes.
    dias_lab = np.floor(rng.beta(1.65, 3.1, n_filas) * max_dias_laborables).astype(int)
    # Línea 0058: Calcula o asigna `exp_sf` para utilizarlo en los pasos siguientes.
    exp_sf = np.floor(rng.gamma(2.15, 34.0, n_filas)).astype(int)
    # Línea 0059: Calcula o asigna `exp_sf` para utilizarlo en los pasos siguientes.
    exp_sf = np.minimum(exp_sf, np.maximum(0, (edad - 18) * 12))

    # Línea 0060: Calcula o asigna `tipos_credito` para utilizarlo en los pasos siguientes.
    tipos_credito = np.array(
        # Línea 0061: Ejecuta esta instrucción como parte del procedimiento descrito.
        [
            # Línea 0062: Ejecuta esta instrucción como parte del procedimiento descrito.
            "CONSUMO",
            # Línea 0063: Ejecuta esta instrucción como parte del procedimiento descrito.
            "VIVIENDA",
            # Línea 0064: Ejecuta esta instrucción como parte del procedimiento descrito.
            "VEHICULAR",
            # Línea 0065: Ejecuta esta instrucción como parte del procedimiento descrito.
            "PRODUCTIVO",
            # Línea 0066: Ejecuta esta instrucción como parte del procedimiento descrito.
            "COMERCIAL",
            # Línea 0067: Ejecuta esta instrucción como parte del procedimiento descrito.
            "MICROCRÉDITO",
            # Línea 0068: Ejecuta esta instrucción como parte del procedimiento descrito.
            "AGROPECUARIO",
            # Línea 0069: Ejecuta esta instrucción como parte del procedimiento descrito.
            "EDUCATIVO",
            # Línea 0070: Ejecuta esta instrucción como parte del procedimiento descrito.
            "TARJETA DE CRÉDITO",
        # Línea 0071: Cierra la estructura, llamada o colección iniciada anteriormente.
        ]
    # Línea 0072: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0073: Calcula o asigna `tipo_credito` para utilizarlo en los pasos siguientes.
    tipo_credito = rng.choice(
        # Línea 0074: Ejecuta esta instrucción como parte del procedimiento descrito.
        tipos_credito,
        # Línea 0075: Ejecuta esta instrucción como parte del procedimiento descrito.
        n_filas,
        # Línea 0076: Calcula o asigna `p` para utilizarlo en los pasos siguientes.
        p=[0.31, 0.10, 0.09, 0.10, 0.07, 0.15, 0.06, 0.03, 0.09],
    # Línea 0077: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0078: Calcula o asigna `terminos` para utilizarlo en los pasos siguientes.
    terminos = {
        # Línea 0079: Registra el valor correspondiente a `CONSUMO` dentro de esta estructura.
        "CONSUMO": (6, 60),
        # Línea 0080: Registra el valor correspondiente a `VIVIENDA` dentro de esta estructura.
        "VIVIENDA": (60, 240),
        # Línea 0081: Registra el valor correspondiente a `VEHICULAR` dentro de esta estructura.
        "VEHICULAR": (12, 84),
        # Línea 0082: Registra el valor correspondiente a `PRODUCTIVO` dentro de esta estructura.
        "PRODUCTIVO": (12, 120),
        # Línea 0083: Registra el valor correspondiente a `COMERCIAL` dentro de esta estructura.
        "COMERCIAL": (6, 84),
        # Línea 0084: Registra el valor correspondiente a `MICROCRÉDITO` dentro de esta estructura.
        "MICROCRÉDITO": (3, 36),
        # Línea 0085: Registra el valor correspondiente a `AGROPECUARIO` dentro de esta estructura.
        "AGROPECUARIO": (6, 60),
        # Línea 0086: Registra el valor correspondiente a `EDUCATIVO` dentro de esta estructura.
        "EDUCATIVO": (6, 84),
        # Línea 0087: Registra el valor correspondiente a `TARJETA DE CRÉDITO` dentro de esta estructura.
        "TARJETA DE CRÉDITO": (1, 48),
    # Línea 0088: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0089: Calcula o asigna `factores_linea` para utilizarlo en los pasos siguientes.
    factores_linea = {
        # Línea 0090: Registra el valor correspondiente a `CONSUMO` dentro de esta estructura.
        "CONSUMO": 6.0,
        # Línea 0091: Registra el valor correspondiente a `VIVIENDA` dentro de esta estructura.
        "VIVIENDA": 45.0,
        # Línea 0092: Registra el valor correspondiente a `VEHICULAR` dentro de esta estructura.
        "VEHICULAR": 18.0,
        # Línea 0093: Registra el valor correspondiente a `PRODUCTIVO` dentro de esta estructura.
        "PRODUCTIVO": 25.0,
        # Línea 0094: Registra el valor correspondiente a `COMERCIAL` dentro de esta estructura.
        "COMERCIAL": 20.0,
        # Línea 0095: Registra el valor correspondiente a `MICROCRÉDITO` dentro de esta estructura.
        "MICROCRÉDITO": 5.0,
        # Línea 0096: Registra el valor correspondiente a `AGROPECUARIO` dentro de esta estructura.
        "AGROPECUARIO": 12.0,
        # Línea 0097: Registra el valor correspondiente a `EDUCATIVO` dentro de esta estructura.
        "EDUCATIVO": 10.0,
        # Línea 0098: Registra el valor correspondiente a `TARJETA DE CRÉDITO` dentro de esta estructura.
        "TARJETA DE CRÉDITO": 4.0,
    # Línea 0099: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0100: Calcula o asigna `tasas_anuales` para utilizarlo en los pasos siguientes.
    tasas_anuales = {
        # Línea 0101: Registra el valor correspondiente a `CONSUMO` dentro de esta estructura.
        "CONSUMO": 0.160,
        # Línea 0102: Registra el valor correspondiente a `VIVIENDA` dentro de esta estructura.
        "VIVIENDA": 0.090,
        # Línea 0103: Registra el valor correspondiente a `VEHICULAR` dentro de esta estructura.
        "VEHICULAR": 0.140,
        # Línea 0104: Registra el valor correspondiente a `PRODUCTIVO` dentro de esta estructura.
        "PRODUCTIVO": 0.120,
        # Línea 0105: Registra el valor correspondiente a `COMERCIAL` dentro de esta estructura.
        "COMERCIAL": 0.130,
        # Línea 0106: Registra el valor correspondiente a `MICROCRÉDITO` dentro de esta estructura.
        "MICROCRÉDITO": 0.220,
        # Línea 0107: Registra el valor correspondiente a `AGROPECUARIO` dentro de esta estructura.
        "AGROPECUARIO": 0.140,
        # Línea 0108: Registra el valor correspondiente a `EDUCATIVO` dentro de esta estructura.
        "EDUCATIVO": 0.100,
        # Línea 0109: Registra el valor correspondiente a `TARJETA DE CRÉDITO` dentro de esta estructura.
        "TARJETA DE CRÉDITO": 0.180,
    # Línea 0110: Cierra la estructura, llamada o colección iniciada anteriormente.
    }

    # Línea 0111: Calcula o asigna `cuotas_restantes` para utilizarlo en los pasos siguientes.
    cuotas_restantes = np.zeros(n_filas, dtype=int)
    # Línea 0112: Calcula o asigna `factor_linea` para utilizarlo en los pasos siguientes.
    factor_linea = np.zeros(n_filas)
    # Línea 0113: Calcula o asigna `tasa_anual` para utilizarlo en los pasos siguientes.
    tasa_anual = np.zeros(n_filas)
    # Línea 0114: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for tipo in tipos_credito:
        # Línea 0115: Ejecuta esta instrucción como parte del procedimiento descrito.
        mascara = tipo_credito == tipo
        # Línea 0116: Calcula o asigna `minimo, maximo` para utilizarlo en los pasos siguientes.
        minimo, maximo = terminos[tipo]
        # Línea 0117: Calcula o asigna `cuotas_restantes[mascara]` para utilizarlo en los pasos siguientes.
        cuotas_restantes[mascara] = rng.integers(minimo, maximo + 1, mascara.sum())
        # Línea 0118: Calcula o asigna `factor_linea[mascara]` para utilizarlo en los pasos siguientes.
        factor_linea[mascara] = factores_linea[tipo]
        # Línea 0119: Calcula o asigna `tasa_anual[mascara]` para utilizarlo en los pasos siguientes.
        tasa_anual[mascara] = tasas_anuales[tipo]

    # Línea 0120: Calcula o asigna `linea_sf` para utilizarlo en los pasos siguientes.
    linea_sf = ingreso * factor_linea * rng.lognormal(0.0, 0.48, n_filas)
    # Línea 0121: Calcula o asigna `linea_sf` para utilizarlo en los pasos siguientes.
    linea_sf = np.round(np.clip(linea_sf, 300, 350_000), 2)

    # Línea 0122: Calcula o asigna `num_entidades_deuda` para utilizarlo en los pasos siguientes.
    num_entidades_deuda = np.clip(1 + rng.poisson(1.05, n_filas), 1, 8)
    # Línea 0123: Calcula o asigna `nuevos_creditos_6m` para utilizarlo en los pasos siguientes.
    nuevos_creditos_6m = np.clip(
        # Línea 0124: Ejecuta esta instrucción como parte del procedimiento descrito.
        rng.poisson(0.22 + 0.09 * (num_entidades_deuda - 1), n_filas), 0, 6
    # Línea 0125: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0126: Calcula o asigna `nuevos_creditos_3m_6m` para utilizarlo en los pasos siguientes.
    nuevos_creditos_3m_6m = np.array(
        # Línea 0127: Calcula o asigna `[rng.integers(0, valor + 1) for valor in nuevos_creditos_6m], dtype` para utilizarlo en los pasos siguientes.
        [rng.integers(0, valor + 1) for valor in nuevos_creditos_6m], dtype=int
    # Línea 0128: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Línea 0129: Calcula o asigna `ahorro_meses` para utilizarlo en los pasos siguientes.
    ahorro_meses = rng.lognormal(
        # Línea 0130: Calcula o asigna `mean` para utilizarlo en los pasos siguientes.
        mean=-0.35 + 0.12 * np.log1p(f_educ), sigma=1.00, size=n_filas
    # Línea 0131: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0132: Calcula o asigna `sin_ahorro` para utilizarlo en los pasos siguientes.
    sin_ahorro = rng.random(n_filas) < 0.14
    # Línea 0133: Calcula o asigna `ahorro_meses[sin_ahorro]` para utilizarlo en los pasos siguientes.
    ahorro_meses[sin_ahorro] = 0.0
    # Línea 0134: Calcula o asigna `saldo_promedio_3m` para utilizarlo en los pasos siguientes.
    saldo_promedio_3m = np.round(np.clip(ingreso * ahorro_meses, 0, 120_000), 2)
    # Línea 0135: Calcula o asigna `ratio_ahorro` para utilizarlo en los pasos siguientes.
    ratio_ahorro = saldo_promedio_3m / ingreso
    # Línea 0136: Calcula o asigna `nivel_ahorro` para utilizarlo en los pasos siguientes.
    nivel_ahorro = np.select(
        # Línea 0137: Ejecuta esta instrucción como parte del procedimiento descrito.
        [
            # Línea 0138: Ejecuta esta instrucción como parte del procedimiento descrito.
            ratio_ahorro < 0.10,
            # Línea 0139: Ejecuta esta instrucción como parte del procedimiento descrito.
            ratio_ahorro < 1.00,
            # Línea 0140: Ejecuta esta instrucción como parte del procedimiento descrito.
            ratio_ahorro < 3.00,
            # Línea 0141: Ejecuta esta instrucción como parte del procedimiento descrito.
            ratio_ahorro < 6.00,
        # Línea 0142: Cierra la estructura, llamada o colección iniciada anteriormente.
        ],
        # Línea 0143: Ejecuta esta instrucción como parte del procedimiento descrito.
        ["SIN AHORRO", "BAJO", "MEDIO", "ALTO"],
        # Línea 0144: Calcula o asigna `default` para utilizarlo en los pasos siguientes.
        default="MUY ALTO",
    # Línea 0145: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Línea 0146: Calcula o asigna `utilizacion_base` para utilizarlo en los pasos siguientes.
    utilizacion_base = rng.beta(1.9, 2.6, n_filas)
    # Línea 0147: Calcula o asigna `presion_crediticia` para utilizarlo en los pasos siguientes.
    presion_crediticia = (
        # Línea 0148: Ejecuta esta instrucción como parte del procedimiento descrito.
        0.12 * (num_entidades_deuda - 1)
        # Línea 0149: Ejecuta esta instrucción como parte del procedimiento descrito.
        + 0.18 * nuevos_creditos_6m
        # Línea 0150: Ejecuta esta instrucción como parte del procedimiento descrito.
        - 0.07 * np.log1p(ratio_ahorro)
    # Línea 0151: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0152: Calcula o asigna `porcentaje_utilizacion_linea` para utilizarlo en los pasos siguientes.
    porcentaje_utilizacion_linea = np.clip(
        # Línea 0153: Ejecuta esta instrucción como parte del procedimiento descrito.
        100 * utilizacion_base + 12 * presion_crediticia + rng.normal(0, 6, n_filas),
        # Línea 0154: Ejecuta esta instrucción como parte del procedimiento descrito.
        1,
        # Línea 0155: Ejecuta esta instrucción como parte del procedimiento descrito.
        135,
    # Línea 0156: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0157: Calcula o asigna `deuda_sf` para utilizarlo en los pasos siguientes.
    deuda_sf = np.round(linea_sf * porcentaje_utilizacion_linea / 100, 2)
    # Línea 0158: Calcula o asigna `saldo_pendiente_credito` para utilizarlo en los pasos siguientes.
    saldo_pendiente_credito = np.round(
        # Línea 0159: Ejecuta esta instrucción como parte del procedimiento descrito.
        deuda_sf * rng.uniform(0.45, 1.00, n_filas), 2
    # Línea 0160: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Línea 0161: Calcula o asigna `tasa_mensual` para utilizarlo en los pasos siguientes.
    tasa_mensual = tasa_anual / 12
    # Línea 0162: Calcula o asigna `factor_pago` para utilizarlo en los pasos siguientes.
    factor_pago = (
        # Línea 0163: Ejecuta esta instrucción como parte del procedimiento descrito.
        tasa_mensual * (1 + tasa_mensual) ** cuotas_restantes
        # Línea 0164: Ejecuta esta instrucción como parte del procedimiento descrito.
        / ((1 + tasa_mensual) ** cuotas_restantes - 1)
    # Línea 0165: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0166: Calcula o asigna `cuota_mensual` para utilizarlo en los pasos siguientes.
    cuota_mensual = saldo_pendiente_credito * factor_pago
    # Línea 0167: Calcula o asigna `cuota_ingreso` para utilizarlo en los pasos siguientes.
    cuota_ingreso = np.clip(100 * cuota_mensual / ingreso, 0.1, 95.0)
    # Línea 0168: Calcula o asigna `dti` para utilizarlo en los pasos siguientes.
    dti = np.clip(100 * deuda_sf / ingreso, 0.1, 5000.0)

    # Línea 0169: Calcula o asigna `riesgo_preliminar` para utilizarlo en los pasos siguientes.
    riesgo_preliminar = (
        # Línea 0170: Ejecuta esta instrucción como parte del procedimiento descrito.
        1.15 * (porcentaje_utilizacion_linea / 100 - 0.45)
        # Línea 0171: Ejecuta esta instrucción como parte del procedimiento descrito.
        + 0.55 * np.log1p(dti / 100)
        # Línea 0172: Ejecuta esta instrucción como parte del procedimiento descrito.
        + 0.85 * np.maximum(cuota_ingreso / 100 - 0.30, 0)
        # Línea 0173: Ejecuta esta instrucción como parte del procedimiento descrito.
        + 0.20 * (num_entidades_deuda - 2)
        # Línea 0174: Ejecuta esta instrucción como parte del procedimiento descrito.
        + 0.28 * nuevos_creditos_6m
        # Línea 0175: Ejecuta esta instrucción como parte del procedimiento descrito.
        - 0.35 * np.log1p(ratio_ahorro)
        # Línea 0176: Ejecuta esta instrucción como parte del procedimiento descrito.
        - 0.18 * np.log1p(ingreso / SBU_2026_USD)
        # Línea 0177: Ejecuta esta instrucción como parte del procedimiento descrito.
        - 0.10 * np.log1p((dias_lab + 1) / 365)
        # Línea 0178: Ejecuta esta instrucción como parte del procedimiento descrito.
        + rng.normal(0, 0.80, n_filas)
    # Línea 0179: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0180: Calcula o asigna `prob_reestructura` para utilizarlo en los pasos siguientes.
    prob_reestructura = _sigmoide(-3.10 + 0.85 * riesgo_preliminar)
    # Línea 0181: Calcula o asigna `reestructura` para utilizarlo en los pasos siguientes.
    reestructura = rng.random(n_filas) < prob_reestructura
    # Línea 0182: Calcula o asigna `riesgo` para utilizarlo en los pasos siguientes.
    riesgo = riesgo_preliminar + 0.85 * reestructura + rng.normal(0, 0.30, n_filas)
    # Línea 0183: Calcula o asigna `mora` para utilizarlo en los pasos siguientes.
    mora = _asignar_mora_ordenada(riesgo)
    # Línea 0184: Calcula o asigna `atraso` para utilizarlo en los pasos siguientes.
    atraso = _dias_segun_clase(rng, mora)

    # Línea 0185: Calcula o asigna `score` para utilizarlo en los pasos siguientes.
    score = np.rint(835 - 82 * riesgo + rng.normal(0, 42, n_filas)).astype(int)
    # Línea 0186: Calcula o asigna `score` para utilizarlo en los pasos siguientes.
    score = np.clip(score, 300, 999)
    # Línea 0187: Calcula o asigna `clasif_sbs` para utilizarlo en los pasos siguientes.
    clasif_sbs = _clasificacion_sbs(rng, mora)

    # Línea 0188: Calcula o asigna `incremento_3m` para utilizarlo en los pasos siguientes.
    incremento_3m = rng.choice([-1, 0, 1], n_filas, p=[0.10, 0.74, 0.16])
    # Línea 0189: Calcula o asigna `clase_hist_3m` para utilizarlo en los pasos siguientes.
    clase_hist_3m = np.maximum(mora, np.clip(mora + incremento_3m, 0, 5))
    # Línea 0190: Calcula o asigna `clase_hist_6m` para utilizarlo en los pasos siguientes.
    clase_hist_6m = np.minimum(
        # Línea 0191: Ejecuta esta instrucción como parte del procedimiento descrito.
        5, clase_hist_3m + (rng.random(n_filas) < (0.12 + 0.04 * mora))
    # Línea 0192: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0193: Calcula o asigna `clase_hist_12m` para utilizarlo en los pasos siguientes.
    clase_hist_12m = np.minimum(
        # Línea 0194: Ejecuta esta instrucción como parte del procedimiento descrito.
        5, clase_hist_6m + (rng.random(n_filas) < (0.16 + 0.04 * mora))
    # Línea 0195: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0196: Calcula o asigna `mora_max_3m` para utilizarlo en los pasos siguientes.
    mora_max_3m = np.maximum(atraso, _dias_segun_clase(rng, clase_hist_3m))
    # Línea 0197: Calcula o asigna `mora_max_6m` para utilizarlo en los pasos siguientes.
    mora_max_6m = np.maximum(mora_max_3m, _dias_segun_clase(rng, clase_hist_6m))
    # Línea 0198: Calcula o asigna `mora_max_12m` para utilizarlo en los pasos siguientes.
    mora_max_12m = np.maximum(mora_max_6m, _dias_segun_clase(rng, clase_hist_12m))

    # El número de eventos aumenta de forma ordenada con el nivel de mora.
    # Línea 0199: Calcula o asigna `min_atrasos_6m` para utilizarlo en los pasos siguientes.
    min_atrasos_6m = np.array([0, 1, 2, 3, 4, 5])[mora]
    # Línea 0200: Calcula o asigna `max_atrasos_6m` para utilizarlo en los pasos siguientes.
    max_atrasos_6m = np.array([2, 4, 6, 8, 10, 12])[mora]
    # Línea 0201: Calcula o asigna `atrasos_6m_12m` para utilizarlo en los pasos siguientes.
    atrasos_6m_12m = np.array(
        # Línea 0202: Ejecuta esta instrucción como parte del procedimiento descrito.
        [
            # Línea 0203: Ejecuta esta instrucción como parte del procedimiento descrito.
            rng.integers(minimo, maximo + 1)
            # Línea 0204: Recorre los elementos requeridos para aplicar la operación de forma repetida.
            for minimo, maximo in zip(min_atrasos_6m, max_atrasos_6m)
        # Línea 0205: Cierra la estructura, llamada o colección iniciada anteriormente.
        ],
        # Línea 0206: Calcula o asigna `dtype` para utilizarlo en los pasos siguientes.
        dtype=int,
    # Línea 0207: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0208: Calcula o asigna `atrasos_12m` para utilizarlo en los pasos siguientes.
    atrasos_12m = np.clip(
        # Línea 0209: Ejecuta esta instrucción como parte del procedimiento descrito.
        atrasos_6m_12m
        # Línea 0210: Ejecuta esta instrucción como parte del procedimiento descrito.
        + rng.poisson(0.15 + 0.18 * clase_hist_12m, n_filas),
        # Línea 0211: Ejecuta esta instrucción como parte del procedimiento descrito.
        0,
        # Línea 0212: Ejecuta esta instrucción como parte del procedimiento descrito.
        24,
    # Línea 0213: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Variación absoluta positiva (%), menor en promedio cuando aumenta la mora.
    # Línea 0214: Calcula o asigna `variacion_depositos_ingresos` para utilizarlo en los pasos siguientes.
    variacion_depositos_ingresos = np.clip(
        # Línea 0215: Calcula o asigna `rng.lognormal(mean` para utilizarlo en los pasos siguientes.
        rng.lognormal(mean=2.35 - 0.12 * mora, sigma=0.60, size=n_filas),
        # Línea 0216: Ejecuta esta instrucción como parte del procedimiento descrito.
        0.01,
        # Línea 0217: Ejecuta esta instrucción como parte del procedimiento descrito.
        120,
    # Línea 0218: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0219: Calcula o asigna `antiguedad_cliente_banco` para utilizarlo en los pasos siguientes.
    antiguedad_cliente_banco = np.floor(rng.gamma(1.9, 27.0, n_filas)).astype(int)
    # Línea 0220: Calcula o asigna `antiguedad_cliente_banco` para utilizarlo en los pasos siguientes.
    antiguedad_cliente_banco = np.minimum(antiguedad_cliente_banco, exp_sf)
    # Línea 0221: Calcula o asigna `numero_productos_banco` para utilizarlo en los pasos siguientes.
    numero_productos_banco = np.clip(
        # Línea 0222: Ejecuta esta instrucción como parte del procedimiento descrito.
        1
        # Línea 0223: Ejecuta esta instrucción como parte del procedimiento descrito.
        + rng.poisson(
            # Línea 0224: Ejecuta esta instrucción como parte del procedimiento descrito.
            0.65
            # Línea 0225: Ejecuta esta instrucción como parte del procedimiento descrito.
            + 0.006 * antiguedad_cliente_banco
            # Línea 0226: Ejecuta esta instrucción como parte del procedimiento descrito.
            + 0.16 * np.log1p(ingreso / SBU_2026_USD),
            # Línea 0227: Ejecuta esta instrucción como parte del procedimiento descrito.
            n_filas,
        # Línea 0228: Cierra la estructura, llamada o colección iniciada anteriormente.
        ),
        # Línea 0229: Ejecuta esta instrucción como parte del procedimiento descrito.
        1,
        # Línea 0230: Ejecuta esta instrucción como parte del procedimiento descrito.
        10,
    # Línea 0231: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Genera cédulas sintéticas válidas antes de aplicar la protección solicitada.
    # Línea 0232: Calcula o asigna `cedulas_sinteticas` para utilizarlo en los pasos siguientes.
    cedulas_sinteticas = _generar_cedulas_ecuatorianas(rng, zona)
    # Crea identificadores únicos terminados en XXX sin exponer la cédula completa.
    # Línea 0233: Calcula o asigna `identificadores` para utilizarlo en los pasos siguientes.
    identificadores = _generar_ids_protegidos(cedulas_sinteticas, semilla + 500)
    # Crea 24 fechas de cierre mensual distribuidas de manera equilibrada.
    # Línea 0234: Calcula o asigna `fecha_reporte` para utilizarlo en los pasos siguientes.
    fecha_reporte = _generar_fechas_reporte(n_filas, semilla + 501)

    # Transforma el nivel numérico interno en la etiqueta textual solicitada.
    # Línea 0235: Calcula o asigna `mora_texto` para utilizarlo en los pasos siguientes.
    mora_texto = np.array([ETIQUETAS_MORA[nivel] for nivel in mora], dtype=object)

    # Línea 0236: Calcula o asigna `df` para utilizarlo en los pasos siguientes.
    df = pd.DataFrame(
        # Línea 0237: Ejecuta esta instrucción como parte del procedimiento descrito.
        {
            # Línea 0238: Registra el valor correspondiente a `ID` dentro de esta estructura.
            "ID": identificadores,
            # Línea 0239: Registra el valor correspondiente a `fecha_reporte` dentro de esta estructura.
            "fecha_reporte": fecha_reporte,
            # Línea 0240: Registra el valor correspondiente a `atraso` dentro de esta estructura.
            "atraso": atraso,
            # Línea 0241: Registra el valor correspondiente a `vivienda` dentro de esta estructura.
            "vivienda": vivienda,
            # Línea 0242: Registra el valor correspondiente a `edad` dentro de esta estructura.
            "edad": edad,
            # Línea 0243: Registra el valor correspondiente a `dias_lab` dentro de esta estructura.
            "dias_lab": dias_lab,
            # Línea 0244: Registra el valor correspondiente a `exp_sf` dentro de esta estructura.
            "exp_sf": exp_sf,
            # Línea 0245: Registra el valor correspondiente a `nivel_ahorro` dentro de esta estructura.
            "nivel_ahorro": nivel_ahorro,
            # Línea 0246: Registra el valor correspondiente a `ingreso` dentro de esta estructura.
            "ingreso": ingreso,
            # Línea 0247: Registra el valor correspondiente a `linea_sf` dentro de esta estructura.
            "linea_sf": linea_sf,
            # Línea 0248: Registra el valor correspondiente a `deuda_sf` dentro de esta estructura.
            "deuda_sf": deuda_sf,
            # Línea 0249: Registra el valor correspondiente a `score` dentro de esta estructura.
            "score": score,
            # Línea 0250: Registra el valor correspondiente a `zona` dentro de esta estructura.
            "zona": zona,
            # Línea 0251: Registra el valor correspondiente a `clasif_sbs` dentro de esta estructura.
            "clasif_sbs": clasif_sbs,
            # Línea 0252: Registra el valor correspondiente a `nivel_educ` dentro de esta estructura.
            "nivel_educ": nivel_educ,
            # Línea 0253: Registra el valor correspondiente a `mora_max_3m` dentro de esta estructura.
            "mora_max_3m": mora_max_3m,
            # Línea 0254: Registra el valor correspondiente a `mora_max_6m` dentro de esta estructura.
            "mora_max_6m": mora_max_6m,
            # Línea 0255: Registra el valor correspondiente a `mora_max_12m` dentro de esta estructura.
            "mora_max_12m": mora_max_12m,
            # Línea 0256: Registra el valor correspondiente a `atrasos_6m_12m` dentro de esta estructura.
            "atrasos_6m_12m": atrasos_6m_12m,
            # Línea 0257: Registra el valor correspondiente a `atrasos_12m` dentro de esta estructura.
            "atrasos_12m": atrasos_12m,
            # Línea 0258: Registra el valor correspondiente a `dti` dentro de esta estructura.
            "dti": dti,
            # Línea 0259: Registra el valor correspondiente a `cuota_ingreso` dentro de esta estructura.
            "cuota_ingreso": cuota_ingreso,
            # Línea 0260: Registra el valor correspondiente a `saldo_promedio_3m` dentro de esta estructura.
            "saldo_promedio_3m": saldo_promedio_3m,
            # Línea 0261: Registra el valor correspondiente a `variacion_depositos_ingresos` dentro de esta estructura.
            "variacion_depositos_ingresos": variacion_depositos_ingresos,
            # Línea 0262: Registra el valor correspondiente a `num_entidades_deuda` dentro de esta estructura.
            "num_entidades_deuda": num_entidades_deuda,
            # Línea 0263: Registra el valor correspondiente a `nuevos_creditos_3m_6m` dentro de esta estructura.
            "nuevos_creditos_3m_6m": nuevos_creditos_3m_6m,
            # Línea 0264: Registra el valor correspondiente a `nuevos_creditos_6m` dentro de esta estructura.
            "nuevos_creditos_6m": nuevos_creditos_6m,
            # Línea 0265: Registra el valor correspondiente a `porcentaje_utilizacion_linea` dentro de esta estructura.
            "porcentaje_utilizacion_linea": porcentaje_utilizacion_linea,
            # Línea 0266: Registra el valor correspondiente a `saldo_pendiente_credito` dentro de esta estructura.
            "saldo_pendiente_credito": saldo_pendiente_credito,
            # Línea 0267: Registra el valor correspondiente a `cuotas_restantes` dentro de esta estructura.
            "cuotas_restantes": cuotas_restantes,
            # Línea 0268: Registra el valor correspondiente a `tipo_credito` dentro de esta estructura.
            "tipo_credito": tipo_credito,
            # Línea 0269: Registra el valor correspondiente a `antiguedad_cliente_banco` dentro de esta estructura.
            "antiguedad_cliente_banco": antiguedad_cliente_banco,
            # Línea 0270: Registra el valor correspondiente a `numero_productos_banco` dentro de esta estructura.
            "numero_productos_banco": numero_productos_banco,
            # Línea 0271: Registra el valor correspondiente a `reestructuraciones_refinanciamientos_previos` dentro de esta estructura.
            "reestructuraciones_refinanciamientos_previos": np.where(
                # Línea 0272: Ejecuta esta instrucción como parte del procedimiento descrito.
                reestructura, "SÍ", "NO"
            # Línea 0273: Cierra la estructura, llamada o colección iniciada anteriormente.
            ),
            # Línea 0274: Registra el valor correspondiente a `mora` dentro de esta estructura.
            "mora": mora_texto,
        # Línea 0275: Cierra la estructura, llamada o colección iniciada anteriormente.
        },
        # Línea 0276: Calcula o asigna `columns` para utilizarlo en los pasos siguientes.
        columns=COLUMNAS,
    # Línea 0277: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Máscaras de ausencia. Se propagan a razones que dependen de otros campos.
    # Línea 0278: Calcula o asigna `mascaras` para utilizarlo en los pasos siguientes.
    mascaras = {
        # Línea 0279: Ejecuta esta instrucción como parte del procedimiento descrito.
        columna: rng.random(n_filas) < tasa
        # Línea 0280: Recorre los elementos requeridos para aplicar la operación de forma repetida.
        for columna, tasa in TASAS_FALTANTES.items()
    # Línea 0281: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0282: Calcula o asigna `mascaras["dti"] |` para utilizarlo en los pasos siguientes.
    mascaras["dti"] |= mascaras["ingreso"] | mascaras["deuda_sf"]
    # Línea 0283: Calcula o asigna `mascaras["cuota_ingreso"] |` para utilizarlo en los pasos siguientes.
    mascaras["cuota_ingreso"] |= (
        # Línea 0284: Ejecuta esta instrucción como parte del procedimiento descrito.
        mascaras["ingreso"]
        # Línea 0285: Ejecuta esta instrucción como parte del procedimiento descrito.
        | mascaras["saldo_pendiente_credito"]
        # Línea 0286: Ejecuta esta instrucción como parte del procedimiento descrito.
        | mascaras["cuotas_restantes"]
    # Línea 0287: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0288: Calcula o asigna `mascaras["porcentaje_utilizacion_linea"] |` para utilizarlo en los pasos siguientes.
    mascaras["porcentaje_utilizacion_linea"] |= (
        # Línea 0289: Ejecuta esta instrucción como parte del procedimiento descrito.
        mascaras["linea_sf"] | mascaras["deuda_sf"]
    # Línea 0290: Cierra la estructura, llamada o colección iniciada anteriormente.
    )

    # Línea 0291: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for columna, mascara in mascaras.items():
        # Línea 0292: Calcula o asigna `df.loc[mascara, columna]` para utilizarlo en los pasos siguientes.
        df.loc[mascara, columna] = pd.NA

    # Tipos anulables: permiten enteros limpios con celdas vacías en el CSV.
    # Línea 0293: Calcula o asigna `df["mora"]` para utilizarlo en los pasos siguientes.
    df["mora"] = df["mora"].astype("string")
    # Línea 0294: Calcula o asigna `df["ID"]` para utilizarlo en los pasos siguientes.
    df["ID"] = df["ID"].astype("string")
    # Línea 0295: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for columna in COLUMNAS_ENTERAS_CON_FALTANTES:
        # Línea 0296: Calcula o asigna `df[columna]` para utilizarlo en los pasos siguientes.
        df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("Int64")
    # Línea 0297: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for columna in COLUMNAS_MONETARIAS + COLUMNAS_PORCENTAJE:
        # Línea 0298: Calcula o asigna `df[columna]` para utilizarlo en los pasos siguientes.
        df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("Float64")
    # Línea 0299: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for columna in set(COLUMNAS) - set(COLUMNAS_ENTERAS_CON_FALTANTES) - {
        # Línea 0300: Ejecuta esta instrucción como parte del procedimiento descrito.
        "mora",
        # Línea 0301: Ejecuta esta instrucción como parte del procedimiento descrito.
        "ID",
        # Línea 0302: Ejecuta esta instrucción como parte del procedimiento descrito.
        *COLUMNAS_MONETARIAS,
        # Línea 0303: Ejecuta esta instrucción como parte del procedimiento descrito.
        *COLUMNAS_PORCENTAJE,
    # Línea 0304: Cierra la estructura, llamada o colección iniciada anteriormente.
    }:
        # Línea 0305: Calcula o asigna `df[columna]` para utilizarlo en los pasos siguientes.
        df[columna] = df[columna].astype("string")

    # Línea 0306: Calcula o asigna `df[COLUMNAS_MONETARIAS]` para utilizarlo en los pasos siguientes.
    df[COLUMNAS_MONETARIAS] = df[COLUMNAS_MONETARIAS].round(2)
    # Línea 0307: Calcula o asigna `df[COLUMNAS_PORCENTAJE]` para utilizarlo en los pasos siguientes.
    df[COLUMNAS_PORCENTAJE] = df[COLUMNAS_PORCENTAJE].round(2)
    # Línea 0308: Devuelve el resultado calculado por la función.
    return df


## 4. Validaciones de calidad


In [4]:
# Línea 0001: Define la función `validar_base` para reutilizar este procedimiento de forma ordenada.
def validar_base(df, n_esperado=N_FILAS):
    # Línea 0002: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df.shape == (n_esperado, len(COLUMNAS)), df.shape
    # Línea 0003: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert list(df.columns) == COLUMNAS
    # Línea 0004: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].is_unique
    # Línea 0005: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].notna().all()
    # Línea 0006: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df.columns[-1] == "mora"
    # Línea 0007: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].str.fullmatch(r"\d{7}XXX").all()
    # Línea 0008: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].str[:2].astype(int).between(1, 24).all()
    # Línea 0009: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].str[2].astype(int).between(0, 5).all()
    # Línea 0010: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["fecha_reporte"].notna().all()
    # Línea 0011: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["fecha_reporte"].nunique() == 24
    # Línea 0012: Calcula o asigna `conteos_mensuales` para utilizarlo en los pasos siguientes.
    conteos_mensuales = df["fecha_reporte"].value_counts()
    # Línea 0013: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert conteos_mensuales.max() - conteos_mensuales.min() <= 1
    # Línea 0014: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["fecha_reporte"].min() == "2024-09-30"
    # Línea 0015: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["fecha_reporte"].max() == "2026-08-31"
    # Línea 0016: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["mora"].notna().all()
    # Línea 0017: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert set(df["mora"].unique()) == set(ETIQUETAS_MORA.values())
    # Línea 0018: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert all(
        # Línea 0019: Ejecuta esta instrucción como parte del procedimiento descrito.
        df[columna].isna().any()
        # Línea 0020: Recorre los elementos requeridos para aplicar la operación de forma repetida.
        for columna in COLUMNAS
        # Línea 0021: Evalúa esta condición antes de ejecutar el bloque asociado.
        if columna not in COLUMNAS_COMPLETAS
    # Línea 0022: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0023: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert all(df[columna].notna().all() for columna in COLUMNAS_COMPLETAS)
    # Línea 0024: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["ID"].isna().sum() == 0
    # Línea 0025: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert df["mora"].isna().sum() == 0
    # Línea 0026: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert set(df["vivienda"].dropna().unique()) == {"PROPIA", "ARRENDADA"}
    # Línea 0027: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert set(df["zona"].dropna().unique()) == {
        # Línea 0028: Ejecuta esta instrucción como parte del procedimiento descrito.
        "QUITO", "GUAYAQUIL", "AMBATO", "IBARRA", "CUENCA", "SANTO DOMINGO"
    # Línea 0029: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0030: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert (df["variacion_depositos_ingresos"].dropna() > 0).all()

    # Línea 0031: Calcula o asigna `intervalos` para utilizarlo en los pasos siguientes.
    intervalos = {
        # Línea 0032: Ejecuta esta instrucción como parte del procedimiento descrito.
        0: (0, 0), 1: (1, 7), 2: (8, 30),
        # Línea 0033: Ejecuta esta instrucción como parte del procedimiento descrito.
        3: (31, 60), 4: (61, 90), 5: (91, 240),
    # Línea 0034: Cierra la estructura, llamada o colección iniciada anteriormente.
    }
    # Línea 0035: Calcula o asigna `filas_atraso` para utilizarlo en los pasos siguientes.
    filas_atraso = df.dropna(subset=["atraso"]).copy()
    # Línea 0036: Calcula o asigna `filas_atraso["nivel_mora"]` para utilizarlo en los pasos siguientes.
    filas_atraso["nivel_mora"] = filas_atraso["mora"].map(NIVEL_MORA_POR_TEXTO)
    # Línea 0037: Recorre los elementos requeridos para aplicar la operación de forma repetida.
    for clase, (minimo, maximo) in intervalos.items():
        # Línea 0038: Ejecuta esta instrucción como parte del procedimiento descrito.
        valores = filas_atraso.loc[filas_atraso["nivel_mora"] == clase, "atraso"]
        # Línea 0039: Comprueba esta condición y detiene la ejecución si no se cumple.
        assert valores.between(minimo, maximo).all()

    # Línea 0040: Calcula o asigna `filas_eventos` para utilizarlo en los pasos siguientes.
    filas_eventos = df.dropna(subset=["atrasos_6m_12m", "atrasos_12m"])
    # Línea 0041: Comprueba esta condición y detiene la ejecución si no se cumple.
    assert (filas_eventos["atrasos_6m_12m"] <= filas_eventos["atrasos_12m"]).all()


# Línea 0042: Define la función `imprimir_resumen` para reutilizar este procedimiento de forma ordenada.
def imprimir_resumen(df, ruta_salida):
    # Línea 0043: Muestra este resultado para facilitar la revisión del proceso.
    print("\nBASE SINTÉTICA GENERADA CORRECTAMENTE")
    # Línea 0044: Muestra este resultado para facilitar la revisión del proceso.
    print(f"Archivo: {ruta_salida}")
    # Línea 0045: Muestra este resultado para facilitar la revisión del proceso.
    print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    # Línea 0046: Muestra este resultado para facilitar la revisión del proceso.
    print("\nDistribución de mora:")
    # Línea 0047: Calcula o asigna `orden_mora` para utilizarlo en los pasos siguientes.
    orden_mora = list(ETIQUETAS_MORA.values())
    # Línea 0048: Calcula o asigna `resumen_mora` para utilizarlo en los pasos siguientes.
    resumen_mora = pd.DataFrame(
        # Línea 0049: Ejecuta esta instrucción como parte del procedimiento descrito.
        {
            # Línea 0050: Registra el valor correspondiente a `n` dentro de esta estructura.
            "n": df["mora"].value_counts().reindex(orden_mora),
            # Línea 0051: Registra el valor correspondiente a `porcentaje` dentro de esta estructura.
            "porcentaje": (
                # Línea 0052: Calcula o asigna `100 * df["mora"].value_counts(normalize` para utilizarlo en los pasos siguientes.
                100 * df["mora"].value_counts(normalize=True).reindex(orden_mora)
            # Línea 0053: Cierra la estructura, llamada o colección iniciada anteriormente.
            ).round(2),
        # Línea 0054: Cierra la estructura, llamada o colección iniciada anteriormente.
        }
    # Línea 0055: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0056: Muestra este resultado para facilitar la revisión del proceso.
    print(resumen_mora.to_string())
    # Línea 0057: Muestra este resultado para facilitar la revisión del proceso.
    print("\nValores faltantes por variable:")
    # Línea 0058: Calcula o asigna `resumen_na` para utilizarlo en los pasos siguientes.
    resumen_na = pd.DataFrame(
        # Línea 0059: Ejecuta esta instrucción como parte del procedimiento descrito.
        {
            # Línea 0060: Registra el valor correspondiente a `faltantes` dentro de esta estructura.
            "faltantes": df.isna().sum(),
            # Línea 0061: Registra el valor correspondiente a `porcentaje` dentro de esta estructura.
            "porcentaje": (100 * df.isna().mean()).round(2),
        # Línea 0062: Cierra la estructura, llamada o colección iniciada anteriormente.
        }
    # Línea 0063: Cierra la estructura, llamada o colección iniciada anteriormente.
    )
    # Línea 0064: Muestra este resultado para facilitar la revisión del proceso.
    print(resumen_na.to_string())


## 5. Ejecución y exportación del CSV


In [5]:
# Línea 0001: Calcula o asigna `df` para utilizarlo en los pasos siguientes.
df = generar_base()
# Línea 0002: Ejecuta esta instrucción como parte del procedimiento descrito.
validar_base(df)
# Línea 0003: Calcula o asigna `carpeta_salida` para utilizarlo en los pasos siguientes.
carpeta_salida = Path("/content") if Path("/content").exists() else Path.cwd()
# Línea 0004: Calcula o asigna `ruta_salida` para utilizarlo en los pasos siguientes.
ruta_salida = carpeta_salida / NOMBRE_SALIDA
# Línea 0005: Exporta el conjunto en formato CSV sin agregar el índice interno de pandas.
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig", na_rep="")
# Línea 0006: Ejecuta esta instrucción como parte del procedimiento descrito.
imprimir_resumen(df, ruta_salida)



BASE SINTÉTICA GENERADA CORRECTAMENTE
Archivo: /workspace/scratch/2a599717615f/base_mora_ecuador_21500_final.csv
Dimensiones: 21,500 filas x 35 columnas

Distribución de mora:
                            n  porcentaje
mora                                     
AL DÍA                  15481        72.0
MORA DE 1 A 7 DÍAS       2150        10.0
MORA DE 8 A 30 DÍAS      1505         7.0
MORA DE 31 A 60 DÍAS      967         4.5
MORA DE 61 A 90 DÍAS      752         3.5
MORA DE MÁS DE 90 DÍAS    645         3.0

Valores faltantes por variable:
                                              faltantes  porcentaje
ID                                                    0        0.00
fecha_reporte                                         0        0.00
atraso                                              431        2.00
vivienda                                            879        4.09
edad                                                207        0.96
dias_lab                                      

## 6. Comprobaciones finales


In [6]:
# Línea 0001: Muestra este resultado para facilitar la revisión del proceso.
print("Dimensiones:", df.shape)
# Línea 0002: Muestra este resultado para facilitar la revisión del proceso.
print("Primera columna:", df.columns[0])
# Línea 0003: Muestra este resultado para facilitar la revisión del proceso.
print("Segunda columna:", df.columns[1])
# Línea 0004: Muestra este resultado para facilitar la revisión del proceso.
print("Última columna:", df.columns[-1])
# Línea 0005: Muestra este resultado para facilitar la revisión del proceso.
print("IDs protegidos y únicos:", df["ID"].str.fullmatch(r"\d{7}XXX").all() and df["ID"].is_unique)
# Línea 0006: Muestra este resultado para facilitar la revisión del proceso.
print("Cantidad de meses:", df["fecha_reporte"].nunique())
# Línea 0007: Muestra este resultado para facilitar la revisión del proceso.
print("Rango de fechas:", df["fecha_reporte"].min(), "a", df["fecha_reporte"].max())
# Línea 0008: Muestra este resultado para facilitar la revisión del proceso.
print("Registros por mes:", df["fecha_reporte"].value_counts().min(), "a", df["fecha_reporte"].value_counts().max())
# Línea 0009: Muestra este resultado para facilitar la revisión del proceso.
print("Clases de mora:", df["mora"].nunique())
# Línea 0010: Muestra este resultado para facilitar la revisión del proceso.
print(df.head(5).to_string(index=False))


Dimensiones: (21500, 35)
Primera columna: ID
Segunda columna: fecha_reporte
Última columna: mora
IDs protegidos y únicos: True
Cantidad de meses: 24
Rango de fechas: 2024-09-30 a 2026-08-31
Registros por mes: 895 a 896
Clases de mora: 6
        ID fecha_reporte  atraso  vivienda  edad  dias_lab  exp_sf nivel_ahorro  ingreso  linea_sf  deuda_sf  score      zona clasif_sbs    nivel_educ  mora_max_3m  mora_max_6m  mora_max_12m  atrasos_6m_12m  atrasos_12m      dti  cuota_ingreso  saldo_promedio_3m  variacion_depositos_ingresos  num_entidades_deuda  nuevos_creditos_3m_6m  nuevos_creditos_6m  porcentaje_utilizacion_linea  saldo_pendiente_credito  cuotas_restantes tipo_credito  antiguedad_cliente_banco  numero_productos_banco reestructuraciones_refinanciamientos_previos                mora
1705559XXX    2025-10-31       0    PROPIA    41      2015      31         BAJO   933.57  16948.85   1317.87    999     QUITO         A3    SECUNDARIA            0            6             6               

## 7. Descarga en Google Colab


In [7]:
# Línea 0001: Inicia un bloque controlado para manejar diferencias entre Colab y el entorno local.
try:
    # Línea 0002: Importa el recurso específico necesario para esta etapa del proceso.
    from google.colab import files
    # Línea 0003: Inicia la descarga del archivo generado cuando se trabaja en Google Colab.
    files.download(str(ruta_salida))
# Línea 0004: Controla el caso en que Google Colab no esté disponible.
except ImportError:
    # Línea 0005: Muestra este resultado para facilitar la revisión del proceso.
    print("Ejecución local: el CSV quedó guardado en", ruta_salida)


Ejecución local: el CSV quedó guardado en /workspace/scratch/2a599717615f/base_mora_ecuador_21500_final.csv
